# Historical SrLC Validation — the headline evaluation

The strongest evaluation claim in the portfolio: check the system against
**real historical FDA regulatory outcomes**, not synthetic data or an LLM's
own judgment.

**Method.** For each drug/reaction pair where FDA's SrLC database shows a
label change *was later required*, we:
1. take the drug's label **as it stood before** that change (`as_of` set to a
   date prior to the change),
2. search that historical label for the reaction,
3. check whether the system would have flagged a **potential gap** (reaction
   absent from the pre-change label),
4. compare against what FDA actually did.

A correct flag = the system, using only pre-change data, surfaces the gap
FDA later acted on.

In [ ]:
import sys
sys.path.append("..")
import pandas as pd
from datetime import timedelta
from pv_assistant.search import MemoryLabelIndex

li = MemoryLabelIndex()
srlc = pd.read_csv("../data/srlc_validation.csv")
srlc

## Was the reaction absent from the pre-change label?

In [ ]:
def pre_change_flag(drug, reaction, change_date):
    """Search the label as of ~6 months before the change; flag a potential
    gap if the reaction terminology is absent from retrieved sections."""
    as_of = (pd.to_datetime(change_date) - timedelta(days=180)).strftime("%Y-%m-%d")
    results = li.search(reaction, drug=drug, as_of=as_of, num_results=5)
    text = " ".join(r["text"].lower() for r in results)
    # crude coverage check: any content word of the reaction present?
    words = [w for w in reaction.lower().replace("(", " ").replace(")", " ").split()
             if len(w) > 4 and w not in ("including","behavior")]
    covered = any(w in text for w in words)
    return {"drug": drug, "reaction": reaction, "as_of": as_of,
            "covered_pre_change": covered,
            "system_flag": "COVERED" if covered else "POTENTIAL_GAP",
            "fda_acted": True}

In [ ]:
rows = [pre_change_flag(r.drug, r.reaction, r.change_date)
        for r in srlc.itertuples()]
res = pd.DataFrame(rows)
res

In [ ]:
anticipated = (res.system_flag == "POTENTIAL_GAP").sum()
total = len(res)
print(f"Anticipated {anticipated}/{total} real FDA label changes "
      f"({anticipated/total:.0%}) from pre-change data alone.")

## The honest limitations (say these before you're asked)

- This validates only against cases where **FDA did act**. It says nothing
  about **false negatives** (gaps the system would miss) beyond this set, and
  nothing about **false positives** (gaps flagged that never warranted
  action) — SrLC has no "nothing happened" rows to test precision against.
- "Validated on true positives" and "validated on precision" are two
  different, only partially overlapping claims. Precision is approximated
  separately via LLM-as-judge on a broader non-SrLC sample (see
  `agent-eval.ipynb`), not proven here.
- The coverage check is a keyword-presence heuristic; a terminology mismatch
  could produce a false gap flag. That is the same synonym problem hybrid
  search addresses, and why every gap is reported as *potential*.

Results feed the Grafana **Historical SrLC Match Rate** panel via
`db.save_srlc_result(...)`.